# Bounded duplicate inspection

A public, synthetic walkthrough of **Plan 003 — Bounded Duplicate Inspection**.

Archiver identifies duplicate *content* by SHA-256 hash. This notebook shows how to inspect the most consequential duplicate groups while keeping the result small, deterministic, and strictly observational. It never selects a keeper, moves a file, or deletes a copy.

## Audience, prerequisites, and goals

This tutorial is for people evaluating a catalog before deciding what—if anything—to do elsewhere. It assumes Python 3.12+, an installed Archiver checkout, and no access to real files.

By the end, you can:

1. Create and scan a tiny synthetic collection.
2. Use `search_duplicate_groups()` to receive complete totals and a bounded detail view.
3. Understand deterministic ordering and why limits matter.
4. Run the equivalent safe CLI inspection.

## Why Plan 003 exists

A catalog may contain thousands of copies of the same bytes. Returning every group and every member is unhelpful for an interactive inspection: it can flood a terminal or notebook and obscure the largest opportunities. Plan 003 separates two needs:

- **Complete aggregates** answer, *how much duplicate content is currently observed?*
- **Bounded details** answer, *which high-impact groups should I inspect first?*

The bound is a presentation/query boundary, not a data-loss policy. Aggregate totals remain complete even when only a few groups or paths are displayed.

In [ ]:
from __future__ import annotations

from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog


def write_group(root: Path, folder: str, names: tuple[str, ...], content: bytes) -> None:
    directory = root / folder
    directory.mkdir(parents=True, exist_ok=True)
    for name in names:
        (directory / name).write_bytes(content)


def format_bytes(size: int) -> str:
    return f'{size:,} B'


workspace = TemporaryDirectory()
root = Path(workspace.name) / 'source'
root.mkdir()
database_path = Path(workspace.name) / 'catalog.sqlite'

# Three duplicate groups plus one unique file.
write_group(root, 'reports', ('final.pdf', 'final-copy.pdf', 'final-backup.pdf'), b'R' * 120)
write_group(root, 'photos', ('summer.jpg', 'summer-export.jpg'), b'P' * 60)
write_group(root, 'notes', ('draft.txt', 'draft-copy.txt'), b'N' * 10)
(root / 'unique.txt').write_bytes(b'unique')

catalog = Catalog.create(database_path)
scan = catalog.scan_directory(root)
print(f'Scanned {scan.files_observed} files with {scan.duplicate_content_group_count} duplicate groups.')

The sample stays entirely in a temporary directory. The catalog records observations; the source files remain unchanged. Now request only the first two groups and the first two members of each group.

In [ ]:
result = catalog.search_duplicate_groups(root, group_limit=2, member_limit=2)
summary = result.summary

print('Complete catalog-wide totals')
print(f'  Duplicate groups: {summary.duplicate_content_group_count}')
print(f'  Duplicate file instances: {summary.duplicate_file_instance_count}')
print(f'  Potential redundant bytes: {format_bytes(summary.potential_redundant_bytes)}')
print()
print(f'Displaying {len(result.groups)} bounded groups:')
for number, group in enumerate(result.groups, start=1):
    paths = [member.relative_path.as_posix() for member in group.members]
    print(f'  {number}. {group.file_instance_count} copies × {format_bytes(group.size_bytes)}')
    print(f'     potential redundant: {format_bytes(group.potential_redundant_bytes)}')
    print(f'     displayed paths: {paths}')

The totals are not capped: this collection has three duplicate groups and seven duplicate file instances, even though only two groups are shown. A group’s potential redundant bytes are `(copies − 1) × content size`. This is a sizing signal, not permission to reclaim space.

Results are deterministic: groups sort by potential redundant bytes (largest first), then full content identity; paths within a group sort by POSIX relative path. Determinism makes review and automation reproducible.

## Expand a view deliberately

If the first bounded view identifies a group worth investigating, request a larger *read-only* view. The catalog still makes no claim that any path is safe to remove.

In [ ]:
expanded = catalog.search_duplicate_groups(root, group_limit=3, member_limit=10)
for number, group in enumerate(expanded.groups, start=1):
    print(f'Group {number}: {group.file_instance_count} copies, {format_bytes(group.potential_redundant_bytes)} potential redundant')
    for member in group.members:
        print(f'  - {member.relative_path.as_posix()}')

## Equivalent CLI use

In a real catalog, first refresh the root, then inspect aggregates or bounded details. `--details` remains observational; the two limits prevent unexpectedly large terminal output.

In [ ]:
import subprocess
import sys

# The CLI owns its catalog in root/.archiver. Its control directory is excluded from scans.
control_database = root / ".archiver" / "catalog.sqlite"
commands = []
if not control_database.exists():
    # if the CLI did not run yet, run it to create the catalog
    commands.append(("init",))
commands.extend(
    (
        ("refresh", "--no-progress"),
        ("duplicates", "--details", "--group-limit", "2", "--member-limit", "2"),
    )
)
for arguments in commands:
    command = [sys.executable, '-c', 'from archiver.cli import main; raise SystemExit(main())', 'catalog', arguments[0], str(root), *arguments[1:]]
    completed = subprocess.run(command, check=True, text=True, capture_output=True)
    if arguments[0] == 'duplicates':
        print(completed.stdout)

In [ ]:
# Safe to rerun
!uv run archiver catalog refresh "{root}" 

In [ ]:
# Safe to rerun
!uv run archiver catalog duplicates "{root}" --details --group-limit 3 --member-limit 1
# or without limits
# uv run archiver catalog duplicates "{root}" --details 

In [ ]:
# no details
!uv run archiver catalog duplicates "{root}" 

## Exercise

Change `group_limit` to `1` and `member_limit` to `1`. Before running the next cell, predict which values remain complete and which become shorter.

In [ ]:
# Answer: summary totals are unchanged; only the displayed group/member projections shrink.
smallest_view = catalog.search_duplicate_groups(root, group_limit=1, member_limit=1)
assert smallest_view.summary == expanded.summary
assert len(smallest_view.groups) == 1
assert len(smallest_view.groups[0].members) == 1
print('Totals are complete; detail is intentionally bounded.')

## Safety boundary and next steps

A duplicate group means only that the observed bytes match. It does **not** establish ownership, retention requirements, backup status, or a safe keeper. Use this feature to prioritize human review or a future explicitly authorized workflow—not to drive deletion.

**Common pitfall:** treating “potential redundant bytes” as reclaimable space. It is an estimate based on content copies, not an action plan.

**Extension:** combine this bounded inspection with a documented review process that records authority and preconditions before any managed operation is considered.

Finally, close the catalog and remove the synthetic workspace.

In [ ]:
catalog.close()
workspace.cleanup()
print('Temporary catalog and synthetic files removed.')